In [6]:
import os
import requests
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

load_dotenv()
load_dotenv(override=True, dotenv_path="../.env.local")

# Claude LLM + Chain
claude_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,
    max_tokens=1024,
)

claude_prompt = ChatPromptTemplate.from_messages([
    ("system", "{system}"),
    ("human",  "{user}")
])

claude_chain = claude_prompt | claude_llm   # prompt -> Claude

def get_claude_response(system: str, user: str) -> str:
    response = claude_chain.invoke({"system": system, "user": user})
    return response.content

# State
class State(TypedDict):
    user_input: str
    route: str
    response: str

# Node 1: Router
def router_node(state: State) -> State:
    route_raw = get_claude_response(
        system="You are a router. Reply with ONLY one word: 'books' or 'claude'.",
        user=f"""
Classify this message:
- 'books'  -> searching for books or book recommendations
- 'claude' -> everything else

Message: "{state['user_input']}"
Answer:"""
    )
    route = "books" if "book" in route_raw.strip().lower() else "claude"
    print(f"  Route -> {route}")
    return {**state, "route": route}

# Node 2: Claude (default)
def claude_node(state: State) -> State:
    print("  Running: claude_chain")
    return {**state, "response": get_claude_response(
        system="You are a helpful assistant.",
        user=state["user_input"]
    )}

# Node 3: Google Books -> Claude formats results
def books_node(state: State) -> State:
    print("  Running: Google Books API -> claude_chain")
    res = requests.get(
        "https://www.googleapis.com/books/v1/volumes",
        params={"q": state["user_input"], "maxResults": 5,
                "key": os.getenv("GOOGLE_BOOKS_API_KEY")}
    )
    books = []
    for item in res.json().get("items", []):
        v = item.get("volumeInfo", {})
        books.append(
            f"Title: {v.get('title','N/A')}\n"
            f"Authors: {', '.join(v.get('authors',['Unknown']))}\n"
            f"Published: {v.get('publishedDate','N/A')}\n"
            f"Preview: {v.get('previewLink','N/A')}\n"
            f"Description: {v.get('description','')[:200]}"
        )
    if not books:
        return {**state, "response": "No books found."}
    return {**state, "response": get_claude_response(
        system="You are a helpful book assistant.",
        user=f"User searched: \"{state['user_input']}\"\n\n" + "\n\n".join(books)
    )}

# Build Graph
def decide_route(state: State) -> Literal["claude", "books"]:
    return state["route"]

graph = StateGraph(State)
graph.add_node("router", router_node)
graph.add_node("claude", claude_node)
graph.add_node("books",  books_node)
graph.set_entry_point("router")
graph.add_conditional_edges("router", decide_route,
    {"claude": "claude", "books": "books"})
graph.add_edge("claude", END)
graph.add_edge("books",  END)
app = graph.compile()

def chat(user_input: str) -> str:
    result = app.invoke({"user_input": user_input, "route": "", "response": ""})
    return result["response"]

print(chat("Find me books about Harry Potter"))
print(chat("What is recursion?"))

  Route -> books
  Running: Google Books API -> claude_chain
# Harry Potter Books Found

I found 5 books matching your search for "Harry Potter":

## 1. **Harry Potter and the Prisoner of Azkaban**
- **Author:** J.K. Rowling
- **Published:** 2004
- **Description:** Harry Potter is lucky to reach the age of thirteen, since he has survived the murderous attacks of the feared Dark Wizard Voldemort three times. But his hopes for a quiet term concentrating on Quidditch...
- [Preview](http://books.google.com/books?id=z4V4PAAACAAJ&dq=Find+me+books+about+Harry+Potter&hl=&cd=1&source=gbs_api)

## 2. **Harry Potter**
- **Author:** J.K. Rowling
- **Published:** 2013
- **Description:** J.K. Rowling's Harry Potter novels are now available in these spectacular new editions, with beautifully designed jackets by renowned woodcut artist Andrew Davidson. This stylish boxed set includes...
- [Preview](http://books.google.com/books?id=IaUyEAAAQBAJ&dq=Find+me+books+about+Harry+Potter&hl=&cd=2&source=gbs_ap